## Using CDHit to create folds according to protein similarity

### 1. Add PDB code to the featurised datasets

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import json

Xtrain = pd.read_csv('/dcs/22/u2243582/cs310/feature_extraction/Xtrainfeatnew.csv', index_col=0)
Xtest = pd.read_csv('/dcs/22/u2243582/cs310/feature_extraction/Xtestfeatnew.csv', index_col=0)
X_features = pd.concat([Xtrain, Xtest])

X_features = X_features.sort_values(by='Index_original')

# Scale features and apply PCA before we split into folds!

scaler = StandardScaler()
scaler.fit(X_features)
X_features_scaled = scaler.transform(X_features)

pca = PCA(0.80) # keep 80% of the variance of the data
pca.fit(X_features_scaled)
X_features_pca = pca.transform(X_features_scaled)

X_features = pd.DataFrame(X_features_pca, index=X_features.index, columns=[f"PC{i+1}" for i in range(X_features_pca.shape[1])]) # convert back to DataFrame

print ("Components:", pca.n_components_ , "Total explained variance:", pca.explained_variance_ratio_.sum())

/dcs/22/u2243582/.local/lib/python3.9/site-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


Components: 6 Total explained variance: 0.8485759168090902


In [2]:
X_fastas = pd.read_csv('/dcs/22/u2243582/cs310/feature_extraction/refined-set-csv.csv')

data = X_features.join(X_fastas['PDB_Code'])
data = data.join(X_fastas['Log_binding'])
display(data)

# toDrop = ['4yx4', '1laf', '4buq', '1k22', '1hmt', '1utn', '4rux', '1bty']

,PC1,PC2,PC3,PC4,PC5,PC6,PDB_Code,Log_binding
Index_original,,,,,,,,
0,-7.852669,2.941040,-9.682766,-3.058416,-1.962969,-0.652606,6ugp,6.16
1,1.110751,4.888805,11.325584,4.570677,4.693178,0.454427,4rdn,5.92
2,6.460603,-2.412404,-3.479704,-3.832504,3.784439,-3.225119,4mo4,4.22
3,-0.121222,0.683820,11.251343,-7.615649,-2.565889,3.422704,3s0b,6.49
4,-6.035023,-6.112950,0.276956,9.758399,7.036210,-2.196562,6r1d,5.40
...,...,...,...,...,...,...,...,...
5244,10.825840,1.542428,1.924121,1.316769,-2.320025,1.101638,6p3t,7.32
5245,26.437339,11.638440,-1.288587,8.136838,0.280988,7.429822,1tx7,4.60
5246,-10.959608,3.955399,4.904754,-10.954458,-5.177558,-2.368372,3ta1,3.25


### 2. Make a .fasta file with the PDB code as the identifier - DON'T NEED TO RUN AGAIN

In [3]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

# Function to write to FASTA file
def write_fasta_from_df(df, fasta_file):
    """
    Write a FASTA file from a DataFrame.
    
    Parameters:
    df : pandas.DataFrame
        DataFrame containing index, pdb_code, and fasta_string columns.
    fasta_file : str
        Output FASTA file name.
    """
    records = [
        SeqRecord(Seq(sequence), id=str(ind), description="")
        for ind, sequence in zip(df.index, df['Protein_FASTA'])
    ]
    
    # Write all records to a FASTA file
    SeqIO.write(records, fasta_file, "fasta")
    print(f"FASTA file written to {fasta_file}")


In [7]:
write_fasta_from_df(data, "pdb.fasta") # X_fastas, "indices.fasta"

FASTA file written to indices.fasta


### 3. I ran processCDHIT() on the .fasta file, returned dictionary where cluster is the key, list of indices is value - download dict from json file

In [49]:
with open("clustered_proteins.json", "r") as json_file:
    cluster_dict = json.load(json_file)

len(cluster_dict) # length of dict should match the output of num of clusters from clustering program

1192

### 4. Convert the dict so that index is the key and cluster id is value

In [50]:
cluster_assignment_dict = {}
for cluster_ind in cluster_dict:
    for pdb in cluster_dict[cluster_ind]:
        cluster_assignment_dict[pdb] = cluster_ind
        
identifiers = list(cluster_assignment_dict.keys())

### 5. Run NRKFold() on the dictionary to get the 5 folds

#### https://github.com/foxtrotmike/BioTools/blob/main/NRKFold.py

In [6]:
import random
import numpy as np
def NRKFold(E,pc,K = 5, shuffle=True):
    """
    Generate non-redundant K-folds for a dataset where each example involves an object 
    that belongs to a certain cluster. This function ensures that no two folds contain 
    objects from the same cluster and aims to distribute the number of examples 
    approximately equally across all folds.

    This is particularly useful in scenarios where data points can naturally group into 
    clusters (e.g., proteins in bioinformatics), and it is important to avoid having 
    similar examples in both the training and validation sets of a particular fold.

    Parameters
    ----------
    E : List
        A list containing identifiers of objects involved in each example.
    pc : Dictionary
        A dictionary mapping each object to its cluster assignment.
    K : Integer, optional
        The number of folds to create. Default is 5.
    shuffle : Boolean, optional
        Determines whether to shuffle the cluster to fold assignments in different runs.
        Default is True.

    Returns
    -------
    List of lists
        A list where each sublist contains the indices of examples in `objects` that belong to a particular fold.

    Example
    -------
    >>> objects = ['obj1', 'obj2', 'obj3', 'obj4', 'obj5', 'obj6', 'obj1']
    >>> clusters = {'obj1': 1, 'obj2': 2, 'obj3': 1, 'obj4': 2, 'obj5': 3, 'obj6': 3}
    >>> folds = NRKFold(objects, clusters, K=2, shuffle=False)
    >>> print(folds)
    Output might be: [[0, 2, 6], [1, 3, 4, 5]]
    Here, objects 'obj1', 'obj3', and 'obj1' (indices 0, 2, 6) are in one fold, 
    and the rest are in another fold, ensuring no fold has objects from the same cluster.
    """
    e = [pc[str(x)] for x in E] #cluster indices of all proteins in the examples
    c2idx={} #indices of examples of each cluster in e
    for i,x in enumerate(e):
        try: 
            c2idx[x].append(i)
        except:
            c2idx[x]=[i]    
    ce = dict([(c,len(c2idx[c])) for c in c2idx]) #counts of examples of different clusters    
    cF = [0]*K; #counts of examples in each fold
    CF = [[] for _ in range(K)]; #clusters in each fold
    F = [[] for _ in range(K)];#indices of examples in each fold
    keys = list(ce.keys())
    if shuffle:
        random.shuffle(keys)
    for k in keys:
        v = ce[k]
        idx = np.argmin(cF)
        cF[idx]+=v
        CF[idx].append(k) #add cluster to fold
        F[idx].extend(c2idx[k])
    return F

In [51]:
folds = NRKFold(identifiers, cluster_assignment_dict)

In [52]:
final_folds = []
for fold in folds:
    temp_fold = []
    for obj in fold:
        pdb = identifiers[obj]
        data_ind = data.index[data['PDB_Code'] == pdb].item()
        temp_fold.append(data_ind)
    final_folds.append(temp_fold)

### 6. Split X and y into folds

In [56]:
# The final NRKfolds:
splits = [
    {
        "train_ix": final_folds[0] + final_folds[1] + final_folds[2] + final_folds[3],
        "test_ix": final_folds[4]
    },
    {
        "train_ix": final_folds[0] + final_folds[1] + final_folds[2] + final_folds[4],
        "test_ix": final_folds[3]
    },
    {
       "train_ix": final_folds[0] + final_folds[1] + final_folds[4] + final_folds[3],
        "test_ix": final_folds[2]
    },
    {
        "train_ix": final_folds[0] + final_folds[4] + final_folds[2] + final_folds[3],
        "test_ix": final_folds[1]
    },
    {
        "train_ix": final_folds[4] + final_folds[1] + final_folds[2] + final_folds[3],
        "test_ix": final_folds[0]
    }
]

### Random Forest

In [57]:
# Use modal hyperparameters from baseline nested cv:
# 'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 40

# Store metrics across folds
pearsonCoeffs = []
pearsonPValues = []
spearmanCoeffs = []
spearmanPValues = []
maeScores = []
varianceScores = []
r2Scores = []

for fold in splits:

    train_ix = np.array(fold["train_ix"])
    test_ix = np.array(fold["test_ix"])
    
    X_train = data.loc[train_ix, 'PC1':'PC6']
    X_test = data.loc[test_ix, 'PC1':'PC6']
    y_train = data.loc[train_ix, 'Log_binding']
    y_test = data.loc[test_ix, 'Log_binding']
    
    randForest = RandomForestRegressor(n_estimators=200, min_samples_split=2, min_samples_leaf=1, max_features='sqrt', max_depth=40, random_state=1)
    randForest.fit(X_train,y_train)
    randForestPred = randForest.predict(X_test)
    
    # Pearson Correlation
    pearsonCoef, pearsonP = pearsonr(y_test, randForestPred)
    pearsonCoeffs.append(pearsonCoef)
    pearsonPValues.append(pearsonP)
    
    # Spearman Correlation
    spearmanCoef, spearmanP = spearmanr(y_test, randForestPred)
    spearmanCoeffs.append(spearmanCoef)
    spearmanPValues.append(spearmanP)
    
    # Mean Absolute Error
    mae = mean_absolute_error(y_test, randForestPred)
    maeScores.append(mae)
    
    # Variance of Errors
    variance = np.var(y_test - randForestPred)
    varianceScores.append(variance)

    # R2 (Coefficient of Determination)
    r2 = r2_score(y_test, randForestPred)
    r2Scores.append(r2)
    
# Calculate mean and standard deviation for each metric
metricsSummary = {
    "Pearson Correlation": (np.mean(pearsonCoeffs), np.std(pearsonCoeffs)),
    "Pearson P-value": (np.mean(pearsonPValues), np.std(pearsonPValues)),
    "Spearman Correlation": (np.mean(spearmanCoeffs), np.std(spearmanCoeffs)),
    "Spearman P-value": (np.mean(spearmanPValues), np.std(spearmanPValues)),
    "Mean Absolute Error": (np.mean(maeScores), np.std(maeScores)),
    "Variance of Errors": (np.mean(varianceScores), np.std(varianceScores)),
    "R2": (np.mean(r2Scores), np.std(r2Scores))
}

# Print metrics summary
for metric, (mean, std) in metricsSummary.items():
    print(f"{metric}: Mean = {mean}, Std = {std}")

Pearson Correlation: Mean = 0.11906786028787689, Std = 0.07658221526929898
Pearson P-value: Mean = 0.18030530830259026, Std = 0.34273878582731726
Spearman Correlation: Mean = 0.10052049197369814, Std = 0.07448782116561256
Spearman P-value: Mean = 0.20243747907792367, Std = 0.2835527745912581
Mean Absolute Error: Mean = 1.591987292189257, Std = 0.1865234739505498
Variance of Errors: Mean = 3.670019454947693, Std = 0.6892764128029808
R2: Mean = -0.07476756798209985, Std = 0.09721551650521033


### SVR

In [58]:
# Use modal hyperparameters from baseline nested cv:
# 'kernel': 'rbf', 'gamma': 0.1, 'epsilon': 0.1, 'C': 4

# Store metrics across folds
pearsonCoeffs = []
pearsonPValues = []
spearmanCoeffs = []
spearmanPValues = []
maeScores = []
varianceScores = []
r2Scores = []

for fold in splits:
    
    train_ix = np.array(fold["train_ix"])
    test_ix = np.array(fold["test_ix"])
    
    X_train = data.loc[train_ix, 'PC1':'PC6']
    X_test = data.loc[test_ix, 'PC1':'PC6']
    y_train = data.loc[train_ix, 'Log_binding']
    y_test = data.loc[test_ix, 'Log_binding']
    
    svRegressor = SVR(kernel='rbf', gamma=0.1, epsilon=0.1, C=4)
    svRegressor.fit(X_train, y_train)
    svRegressorPred = svRegressor.predict(X_test)
    
    # Pearson Correlation
    pearsonCoef, pearsonP = pearsonr(y_test, svRegressorPred)
    pearsonCoeffs.append(pearsonCoef)
    pearsonPValues.append(pearsonP)
    
    # Spearman Correlation
    spearmanCoef, spearmanP = spearmanr(y_test, svRegressorPred)
    spearmanCoeffs.append(spearmanCoef)
    spearmanPValues.append(spearmanP)
    
    # Mean Absolute Error
    mae = mean_absolute_error(y_test, svRegressorPred)
    maeScores.append(mae)
    
    # Variance of Errors
    variance = np.var(y_test - svRegressorPred)
    varianceScores.append(variance)

    # R2 (Coefficient of Determination)
    r2 = r2_score(y_test, svRegressorPred)
    r2Scores.append(r2)
    
# Calculate mean and standard deviation for each metric
metricsSummary = {
    "Pearson Correlation": (np.mean(pearsonCoeffs), np.std(pearsonCoeffs)),
    "Pearson P-value": (np.mean(pearsonPValues), np.std(pearsonPValues)),
    "Spearman Correlation": (np.mean(spearmanCoeffs), np.std(spearmanCoeffs)),
    "Spearman P-value": (np.mean(spearmanPValues), np.std(spearmanPValues)),
    "Mean Absolute Error": (np.mean(maeScores), np.std(maeScores)),
    "Variance of Errors": (np.mean(varianceScores), np.std(varianceScores)),
    "R2": (np.mean(r2Scores), np.std(r2Scores))
}

# Print metrics summary
for metric, (mean, std) in metricsSummary.items():
    print(f"{metric}: Mean = {mean}, Std = {std}")

Pearson Correlation: Mean = 0.11545851028737097, Std = 0.052232159854301886
Pearson P-value: Mean = 0.02659893053071639, Std = 0.04378251554028365
Spearman Correlation: Mean = 0.08763885946051513, Std = 0.051117126503609674
Spearman P-value: Mean = 0.126830634693689, Std = 0.15402842887301144
Mean Absolute Error: Mean = 1.6017227152680626, Std = 0.17162248380076642
Variance of Errors: Mean = 3.7575652575209175, Std = 0.6044815790616724
R2: Mean = -0.0963352608693504, Std = 0.06594446790213111


### XGBoost

In [59]:
# Use modal hyperparameters from baseline nested cv:
# 'subsample': 1, 'reg_lambda': 10, 'reg_alpha': 1, 'max_depth': 9, 'learning_rate': 0.1, 'gamma': 0, 'colsample_bytree': 1

# Store metrics across folds
pearsonCoeffs = []
pearsonPValues = []
spearmanCoeffs = []
spearmanPValues = []
maeScores = []
varianceScores = []
r2Scores = []

for fold in splits:
    
    train_ix = np.array(fold["train_ix"])
    test_ix = np.array(fold["test_ix"])
    
    X_train = data.loc[train_ix, 'PC1':'PC6']
    X_test = data.loc[test_ix, 'PC1':'PC6']
    y_train = data.loc[train_ix, 'Log_binding']
    y_test = data.loc[test_ix, 'Log_binding']
    
    xgReg = XGBRegressor(subsample=1, reg_lambda=10, reg_alpha=1, max_depth=9, learning_rate=0.1, gamma=0, colsample_bytree=1, random_state=1)
    xgReg.fit(X_train, y_train)
    xgPred = xgReg.predict(X_test)
    
    # Pearson Correlation
    pearsonCoef, pearsonP = pearsonr(y_test, xgPred)
    pearsonCoeffs.append(pearsonCoef)
    pearsonPValues.append(pearsonP)
    
    # Spearman Correlation
    spearmanCoef, spearmanP = spearmanr(y_test, xgPred)
    spearmanCoeffs.append(spearmanCoef)
    spearmanPValues.append(spearmanP)
    
    # Mean Absolute Error
    mae = mean_absolute_error(y_test, xgPred)
    maeScores.append(mae)
    
    # Variance of Errors
    variance = np.var(y_test - xgPred)
    varianceScores.append(variance)

    # R2 (Coefficient of Determination)
    r2 = r2_score(y_test, xgPred)
    r2Scores.append(r2)
    
# Calculate mean and standard deviation for each metric
metricsSummary = {
    "Pearson Correlation": (np.mean(pearsonCoeffs), np.std(pearsonCoeffs)),
    "Pearson P-value": (np.mean(pearsonPValues), np.std(pearsonPValues)),
    "Spearman Correlation": (np.mean(spearmanCoeffs), np.std(spearmanCoeffs)),
    "Spearman P-value": (np.mean(spearmanPValues), np.std(spearmanPValues)),
    "Mean Absolute Error": (np.mean(maeScores), np.std(maeScores)),
    "Variance of Errors": (np.mean(varianceScores), np.std(varianceScores)),
    "R2": (np.mean(r2Scores), np.std(r2Scores))
}

# Print metrics summary
for metric, (mean, std) in metricsSummary.items():
    print(f"{metric}: Mean = {mean}, Std = {std}")

Pearson Correlation: Mean = 0.12439367338900698, Std = 0.05207010525631229
Pearson P-value: Mean = 0.012741258492512383, Std = 0.024978397930506564
Spearman Correlation: Mean = 0.13121455244672314, Std = 0.053102496923003534
Spearman P-value: Mean = 0.010644608672512903, Std = 0.020587856671237026
Mean Absolute Error: Mean = 1.6166335888076895, Std = 0.1520590495250976
Variance of Errors: Mean = 3.8182721059135134, Std = 0.5514251944618597
R2: Mean = -0.11977771645468316, Std = 0.06346367760064192
